<a href="https://colab.research.google.com/github/ahmedinB/MDrone/blob/main/MKDrone_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import the opencv library
import cv2 , math, torch
import torch.nn as nn
import torch.optim as optim
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from zoedepth.models.builder import build_model
from zoedepth.utils.config import get_config
from zoedepth.utils.misc import colorize
from zoedepth.utils.misc import save_raw_16bit
from zoedepth.utils.misc import get_image_from_url
from PIL import Image

In [ ]:
# define a video capture object
vid = cv2.VideoCapture(0)
vid.set(3,640)
vid.set(4,540)

In [ ]:
model = YOLO('weights/best.pt')
# ZoeD_NK
conf = get_config("zoedepth_nk", "infer")
model_zoe_nk = build_model(conf)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
zoe = model_zoe_nk.to(DEVICE)

In [ ]:
# load MLP model
class LocalizationMLP(nn.Module):
    def __init__(self):
        super(LocalizationMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Load the best model
model_path = 'D:/anaconda/envs/yol/mlp/best_localization_model_100.pth'  # Adjust the path as necessary
mlp = LocalizationMLP()
mlp.load_state_dict(torch.load(model_path))

In [ ]:
mlp.eval()  # Set the model to evaluation mode

while(True):

    # Capture the video frame
    # by frame
    ret, frame = vid.read()

    result = model(frame, stream = True)

    depth = zoe.infer_pil(frame)
    # print(depth)
    for r in result:
        boxes = r.boxes

        for box in boxes:
            # bounding box
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            cv2.rectangle(frame, (x1,y1) , (x2, y2), (255, 0, 255), 3)

            org = [x1, y1]
            font = cv2.FONT_HERSHEY_SIMPLEX
            fontscale = 1
            color = (255,0,0)
            thickness = 2
            x, y = box.xywh[0][:2]

            # print("(",x.item(), y.item(), depth[int(y.item()),int(x.item())],")")
            loc = [x.item(), y.item(), depth[int(y.item()),int(x.item())]]
            # print("loc before the MLP", loc)
            # Run MLP model

            loc = torch.tensor(loc)
            loc = mlp(loc)
            # Convert outputs to numpy for easier handling
            loc = loc.detach().numpy().squeeze()
            print(loc)

            loc = [str(float("{:.2f}".format(i))) for i in loc]
            label = "Drone " + ",".join(loc)
            cv2.putText(frame, label, org, font,fontscale, color, thickness)


    # Display the resulting frame
    cv2.imshow('frame', frame)

    # the 'q' button is set as the
    # quitting button you may use any
    # desired button of your choice
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# After the loop release the cap object
vid.release()
# Destroy all the windows
cv2.destroyAllWindows()